# RefineNet（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
セマンティックセグメンテーションにおいて，バックボーンの深い層（低解像度・高次特徴）と浅い層（高解像度・低次特徴）を，FPN/RetinaNetのようなtop-down経路で段階的に統合していく手法であるRefineNetを，PASCAL VOC 2007データセットを用いて実装する．PSPNetやDeepLabのように単一の解像度で文脈情報を集約するのではなく，複数解像度の特徴をRefineNetブロック（Residual Conv Unit・Multi-Resolution Fusion・Chained Residual Poolingの3つのサブモジュールからなる）で段階的に融合していく設計を理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import resnet50, ResNet50_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．20クラスの物体について画素単位のクラスラベルが付与されており，学習用（trainval）422枚，評価用（test）210枚という少ない枚数から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## RefineNet
RefineNetは，事前学習済みCNNの複数の解像度の特徴マップを，深い層から浅い層へ向かって段階的に統合していくことで，高解像度かつ高次のセグメンテーション結果を得る手法です．PSPNetやDeepLabのように最後の1つの解像度だけに文脈集約モジュールを適用するのではなく，**複数解像度の特徴を「上から下へ」段階的に統合していく**構造である点が特徴です（FPN/RetinaNetのtop-downパスに近い考え方ですが，統合の際にRCU・Multi-Resolution Fusion・Chained Residual Poolingという独自のモジュールを使う点が異なります）．

### バックボーン（ResNet50）
事前学習済みResNet50の4つのステージ（`layer1`〜`layer4`）の出力を利用します．各ステージの出力は，入力に対してそれぞれ1/4, 1/8, 1/16, 1/32の解像度を持ちます．各ステージの出力チャンネル数（256, 512, 1024, 2048）はまちまちなので，後段のRefineNetブロックで扱いやすいよう，1x1畳み込みで共通のチャンネル数（`feature_dim`）に揃えておきます．

In [ ]:
feature_dim = 256  # 各backboneステージの出力を統一するチャンネル数（RefineNetブロックの共通チャンネル数）

# ResNet50の各ステージの出力チャンネル数とstride（参考）
# layer1: channels=256,  stride=4
# layer2: channels=512,  stride=8
# layer3: channels=1024, stride=16
# layer4: channels=2048, stride=32

### Residual Conv Unit (RCU)
RefineNetブロックの各入力パスに最初に適用するモジュールです．通常の残差ブロックと同様の考え方で，`ReLU`→`3x3畳み込み`を2回適用した出力を，元の入力に加算します（原論文のRCUはBatchNormを持たないシンプルな残差ブロックです）．

In [ ]:
class RCU(nn.Module):
    """Residual Conv Unit：3x3畳み込み×2＋skip接続からなる単純な残差ブロック"""
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, padding=1)

    def forward(self, x):
        h = self.conv1(F.relu(x))
        h = self.conv2(F.relu(h))
        return x + h

### Multi-Resolution Fusion (MRF)
複数解像度の特徴を統合するモジュールです．各入力に3x3畳み込みを適用してチャンネル数・表現を整えたのち，入力のうち最も高い解像度にバイリニア補間でアップサンプリングして全て加算します．最も深い層（`layer4`）の段は入力が1つしかないため，このモジュールは使用しません．

In [ ]:
class MultiResolutionFusion(nn.Module):
    """複数解像度の特徴を，最も高い解像度に揃えて加算するモジュール"""
    def __init__(self, dim, num_inputs):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv2d(dim, dim, kernel_size=3, padding=1) for _ in range(num_inputs)])

    def forward(self, xs):
        target_size = max((x.shape[-2:] for x in xs), key=lambda s: s[0] * s[1])  # 最も高い解像度に合わせる
        out = 0
        for x, conv in zip(xs, self.convs):
            h = conv(x)
            if h.shape[-2:] != target_size:
                h = F.interpolate(h, size=target_size, mode='bilinear', align_corners=False)
            out = out + h
        return out

### Chained Residual Pooling (CRP)
Multi-Resolution Fusionで統合した特徴に対して適用するモジュールです．`MaxPool2d`（カーネルサイズ5，stride 1）と3x3畳み込みからなるブロックを連鎖的（chained）に繰り返し適用し，各段の出力を全て加算していきます．プーリングを繰り返すことでパラメータ数をほとんど増やさずに広い受容野を確保でき，背景など広範囲の文脈情報を効率よく集約できます．

In [ ]:
class ChainedResidualPooling(nn.Module):
    """MaxPoolと畳み込みを連鎖的に適用し，各段の出力を加算して文脈情報を集約するモジュール"""
    def __init__(self, dim, n_stages=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(dim, dim, kernel_size=3, padding=1) for _ in range(n_stages)])

    def forward(self, x):
        x = F.relu(x)
        out = x
        for conv in self.convs:
            x = conv(self.pool(x))  # 直前のプーリング結果をさらにプーリングする（chained）
            out = out + x  # 各段の出力を加算していく
        return out

### RefineNetブロック
以上の3つのモジュールを組み合わせ，1つの解像度段を処理するRefineNetブロックを定義します．各入力（前段のRefineNet出力とbackboneの特徴）にRCUを適用したのち，Multi-Resolution Fusionで統合し，Chained Residual Poolingで文脈情報を集約，最後にもう一度RCUを適用して出力します．入力が1つ（最初の`layer4`の段）の場合はMulti-Resolution Fusionを省略します．

In [ ]:
class RefineNetBlock(nn.Module):
    """RCU -> Multi-Resolution Fusion -> Chained Residual Pooling -> RCU からなる1解像度段分のブロック"""
    def __init__(self, dim, num_inputs):
        super().__init__()
        self.rcu_in = nn.ModuleList([nn.Sequential(RCU(dim), RCU(dim)) for _ in range(num_inputs)])
        self.mrf = MultiResolutionFusion(dim, num_inputs) if num_inputs > 1 else None
        self.crp = ChainedResidualPooling(dim)
        self.rcu_out = RCU(dim)

    def forward(self, xs):
        xs = [rcu(x) for rcu, x in zip(self.rcu_in, xs)]
        x = self.mrf(xs) if self.mrf is not None else xs[0]
        x = self.crp(x)
        x = self.rcu_out(x)
        return x

### RefineNet全体の構成
ResNet50の`layer1`〜`layer4`の出力を1x1畳み込みで`feature_dim`チャンネルに揃えたのち，最も解像度の低い`layer4`から`layer1`へ向かって，RefineNetブロックを順に適用していきます．各段では，1つ前の段のRefineNetブロックの出力と，対応するbackboneステージの特徴の2つを入力とします（`layer4`の段のみ，backboneの特徴1つだけを入力とします）．最終的に最も高解像度（`layer1`，入力の1/4サイズ）のRefineNet出力を1x1畳み込みでクラス数チャンネルに変換し，入力画像と同じサイズにバイリニア補間でアップサンプリングします．

In [ ]:
class RefineNet(nn.Module):
    """ResNet50をバックボーンとしたRefineNet"""
    def __init__(self, n_class=n_class, feature_dim=feature_dim):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # stride 4,  256 channels
        self.layer2 = resnet.layer2  # stride 8,  512 channels
        self.layer3 = resnet.layer3  # stride 16, 1024 channels
        self.layer4 = resnet.layer4  # stride 32, 2048 channels

        self.adapt1 = nn.Conv2d(256, feature_dim, kernel_size=1)
        self.adapt2 = nn.Conv2d(512, feature_dim, kernel_size=1)
        self.adapt3 = nn.Conv2d(1024, feature_dim, kernel_size=1)
        self.adapt4 = nn.Conv2d(2048, feature_dim, kernel_size=1)

        # layer4の段は入力が1つ（backboneの特徴のみ），それ以外は「前段の出力」と「backboneの特徴」の2つ
        self.refine4 = RefineNetBlock(feature_dim, num_inputs=1)
        self.refine3 = RefineNetBlock(feature_dim, num_inputs=2)
        self.refine2 = RefineNetBlock(feature_dim, num_inputs=2)
        self.refine1 = RefineNetBlock(feature_dim, num_inputs=2)

        self.classifier = nn.Conv2d(feature_dim, n_class, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[-2:]

        x0 = self.layer0(x)
        f1 = self.layer1(x0)  # stride 4,  256 channels
        f2 = self.layer2(f1)  # stride 8,  512 channels
        f3 = self.layer3(f2)  # stride 16, 1024 channels
        f4 = self.layer4(f3)  # stride 32, 2048 channels

        c1, c2, c3, c4 = self.adapt1(f1), self.adapt2(f2), self.adapt3(f3), self.adapt4(f4)

        # layer4（最も低解像度）から layer1（最も高解像度）へ向かって段階的に統合していく
        r4 = self.refine4([c4])
        r3 = self.refine3([r4, c3])
        r2 = self.refine2([r3, c2])
        r1 = self.refine1([r2, c1])

        out = self.classifier(r1)
        out = F.interpolate(out, size=input_size, mode='bilinear', align_corners=False)
        return out

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
RefineNetを構築し，最適化手法としてAdamを設定します．バックボーンには事前学習済みの重みを使用しているため，スクラッチ学習の分類ノートブック群（`classification/`）よりも小さい学習率を用います．

In [ ]:
model = RefineNet(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてRefineNetを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのRefineNetとの違い

| 項目 | オリジナル論文 (Lin et al., 2017) | 本ノートブック |
| :-- | :-- | :-- |
| バックボーン | ResNet101 / ResNeXt101など，より深いモデルを使用 | 事前学習済みResNet50を使用（計算量を抑えるため） |
| 構成のバリエーション | 4-cascade構成に加え，2-cascadeやマルチパス入力構成など複数のバリエーションを提案 | 4段構成（`layer4`→`layer1`）の基本形（4-cascade）のみを実装 |
| データ拡張・評価方法 | マルチスケールでの学習・推論やアンサンブル評価を実施 | 固定サイズへのリサイズのみ（データ拡張・マルチスケール評価なし） |
| 学習データ量 | PASCAL VOCの拡張データ（SBD）を含む約1万枚以上 | VOC2007のtrainvalのみ（422枚） |

## 課題

1. `feature_dim`やChained Residual Poolingの繰り返し回数（`n_stages`）を変更し，mIoUや学習速度への影響を確認してください．
2. RefineNetBlockの入力パスに適用するRCUの回数（現在は各入力2回）を1回や3回に変更し，精度がどのように変化するか確認してください．
3. バックボーンをResNet50から別のモデル（例えば`torchvision.models.resnet34`）に変更する場合，`adapt1`〜`adapt4`の入力チャンネル数をどのように変更する必要があるか考えてみてください．